In [ ]:
# 04_ramp_shock_baseline.ipynb -- LightGBM baseline for the 1-4-slot-lead-time
# ramp-shock target (same feature pipeline as 03_violation_baseline.ipynb)
# !pip install lightgbm -q

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve

import features as f

TARGET = "ramp_lead"

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
feat_df = f.build_feature_table(scada)

df = feat_df.dropna(subset=[TARGET]).copy()

# --- Time-aware split (same as 03_violation_baseline.ipynb) ---
train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
test = df[df["date"] >= "2026-01-01"]

print("train:", train.shape, "val:", val.shape, "test:", test.shape)
print("event rate train/val/test:", train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean())

X_train, y_train = train[f.FEATURE_COLS], train[TARGET]
X_val, y_val = val[f.FEATURE_COLS], val[TARGET]
X_test, y_test = test[f.FEATURE_COLS], test[TARGET]

# NOTE (2026-07-11): NOT using scale_pos_weight -- see features.py's
# scale_pos_weight() docstring and 03_violation_baseline.ipynb for the full story.
model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)
print(f"\nbest_iteration_: {model.best_iteration_}")

proba_test = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, proba_test)
base_rate = y_test.mean()
print(f"PR-AUC: {pr_auc:.4f}  (random baseline = base rate = {base_rate:.4f})")

preds_05 = (proba_test >= 0.5).astype(int)
print(f"F1 @ 0.5 threshold: {f1_score(y_test, preds_05):.4f}")

precision, recall, thresh = precision_recall_curve(y_test, proba_test)
f1s = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = np.nanargmax(f1s[:-1])
print(f"Best-F1 operating point: F1={f1s[best_idx]:.4f} at threshold={thresh[best_idx]:.4f} "
      f"(precision={precision[best_idx]:.4f}, recall={recall[best_idx]:.4f})")

idx95 = np.where(precision[:-1] >= 0.95)[0]
recall_at_95p = recall[idx95].max() if len(idx95) else 0.0
print(f"Recall at >=95% precision: {recall_at_95p:.4f}")

importance = pd.Series(model.feature_importances_, index=f.FEATURE_COLS).sort_values(ascending=False)
print("\nTop 15 features:\n", importance.head(15))

# --- Results (verified 2026-07-11, LightGBM 4.6.0; third version of this notebook's
#     numbers -- see history below) ---
# PR-AUC 0.7446 vs a random/base-rate baseline of 0.1793 -- 4.15x lift over chance.
# F1@0.5 = 0.5577. Best-F1 operating point (threshold ~0.22): F1=0.6785,
# precision=62.8%, recall=73.8% -- both PR-AUC and best-F1 improved again from the
# previous version. Recall at >=95% precision = 0.1966 -- this is the one metric that
# went DOWN slightly from the previous version (was 0.2238), worth reporting honestly
# rather than only highlighting the metrics that improved.
#
# History of this notebook's numbers: (1) original PR-AUC 0.7140 with scale_pos_weight;
# (2) removing it + adding solar-volatility features raised it to 0.7248; (3) THIS
# version: removing share_res_pct and the 11 corridor/cross-border columns from the
# feature set (see 03_violation_baseline.ipynb for the full reasoning -- these are
# whole-day aggregates broadcast identically to all 96 slots, tested as a potential
# leakage concern, found instead to just be adding noise) raised it further to 0.7446.
# Top features are now hour, demand_delta_mw, month, solar_delta_mw, and
# demand_met_mw_lag3 -- solar and demand-trajectory features dominate, consistent with
# 01_eda.ipynb's sunrise/sunset clustering finding. Corridor columns no longer appear
# at all, since they're no longer in the feature set -- Era 2's daily-resolution
# corridor-flow finding remains a separate, valid, unaffected result (see
# 00_era2_daily_correlation.ipynb), it just isn't what drives this live classifier.
#
# This target remains markedly easier to predict with lead time than frequency
# violation (see 03_violation_baseline.ipynb, PR-AUC 0.1186 even after the same fixes)
# -- plausibly because a ramp-shock is a direct, mechanical property of the
# demand/generation trajectory itself, while a frequency violation is a downstream
# consequence that depends on how well AGC/reserves absorb a given ramp, adding a layer
# of noise the raw features here don't fully capture.
